# MapReduce-Reloaded: the demo, end to end

This notebook runs the demo in [`docs/demo.md`](../docs/demo.md) against the
Compose cluster, one claim per section. Sections 7, 8, 10, 11 and 13 check
their claims with `assert`, and 13 checks section 5's output byte for byte;
section 9 checks that its job succeeds, not its counts. The others show
what the cluster did, and any command that fails stops the run:

| § | What it shows | Paper |
|---|---|---|
| 2–5 | Four one-CPU workers; upload, submit and read word count from the host | §2, §3.1 |
| 6 | The answer, plotted | |
| 7 | A job file: map and reduce as Go source, interpreted by the workers | §2 |
| 8 | The combiner: fewer bytes shuffled, the same output | §4.3 |
| 9 | A second program, URL access frequency | §2.3 |
| 10 | A worker killed mid-job: its tasks run again elsewhere | §3.3 |
| 11 | The coordinator restarted mid-job: the job resumes from its log | beyond the paper |
| 12 | The coordinator's metrics | §4.8, §4.9 |
| 13 | Local mode agrees with the cluster byte for byte | §4.7 |
| 14 | The evaluation at a glance: the measured results | §5 |
| 15 | If a cell fails: what to check and run | |

**How the system fits together.** The `mr` client on the host submits a
job to the coordinator, the paper's master, over gRPC. Four workers, each a
container limited to one CPU, pull map and reduce tasks from the
coordinator, send it heartbeats, and fetch each other's map output directly
for the shuffle, so no data passes through the coordinator. Input and output
live in the host's `data/` directory, which every container sees at
`/data`; the coordinator writes every decision to a write-ahead log, so it
can crash and resume. [`design.md`](../design.md#components) has the details.

```text
mr (host) ──submit, status──▶ coordinator ◀──pull tasks, heartbeats── workers 1–4
                              (write-ahead log)                       ⇅ shuffle, worker to worker
          input and output: data/ on the host, /data in every container
```

**Before you start**, on the VM, from the repository root:

```sh
make cluster-up mr      # the cluster and the client
make upload-example     # the corpus and a 10 MB web log
python/.venv/bin/pip install -e 'python[notebook]'   # Jupyter and matplotlib
```

Then open it with `python/.venv/bin/jupyter notebook examples/demo.ipynb`, or
run every cell without a browser:

```sh
python/.venv/bin/jupyter nbconvert --to notebook --execute --inplace examples/demo.ipynb
```

*Kernel → Restart & Run All* takes about two minutes on the course VM, almost
all of it in sections 10 and 11, which sort 1 GiB each (the input is
generated on first use: about 20 seconds and 1 GB of disk). Set
`KILL_WORKER = False` or `RESTART_COORDINATOR = False` in the setup cell to
skip either. Rerunning from the top is safe once the previous run has
finished or stopped: each run uses timestamped output directories, the last
cell removes them, and the setup cell removes any an interrupted run left.
Run one copy at a time against a cluster: the setup cell would also remove
the directories of a copy still running. The status page is at
<http://127.0.0.1:18780/>; keep it open beside the notebook. If a cell
fails, section 15 says what to check.


## 1. Setup


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

from IPython.display import Image, Markdown, display

# Sections 10 and 11 each sort 1 GiB: one kills a worker, the other
# restarts the coordinator. False skips a section.
KILL_WORKER = True
RESTART_COORDINATOR = True


def find_root():
    """The repository root: MR_ROOT, or the first of the kernel's working
    directory and its parents that holds deploy/compose. Jupyter starts the
    kernel in examples/, the notebook's directory."""
    if "MR_ROOT" in os.environ:
        return Path(os.environ["MR_ROOT"])
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "deploy" / "compose" / "compose.yaml").exists():
            return path
    raise RuntimeError("not inside the repository: set MR_ROOT to its path")


ROOT = find_root()
MR = ROOT / "bin" / "mr"
RUN = f"{int(time.time())}-{os.getpid()}"  # unique per run, even within a second
CORPUS = f"/data/input/notebook-corpus-{RUN}"
COMPOSE = ["docker", "compose", "-p", "mapreduce", "-f", "deploy/compose/compose.yaml"]
METRICS = "http://127.0.0.1:18780/metrics"
STATUS = "http://127.0.0.1:18780/status.json"  # the status page, as data
EXPECTED = b"".join(p.read_bytes() for p in sorted((ROOT / "testdata/corpus/expected").glob("part-r-*")))
JOBS = {}  # this run's job ids and what each ran, for section 12
# A run stopped by a failing cell never reaches the last section's cleanup,
# so clear what earlier runs left before this one adds more.
for old in [*(ROOT / "data/out").glob("notebook-*"), *(ROOT / "data/input").glob("notebook-corpus-*")]:
    shutil.rmtree(old, ignore_errors=True)
SORT_INPUT = ROOT / "data/input/sort-1g.txt"
# The project's chart colors (bench/plot-experiments.py): blue is the
# system as shipped, orange the changed condition, text in ink, not color.
BLUE, ORANGE, INK, INK2 = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"


def run(*args, check=True, timeout=None):
    """Runs a command in the repository root and returns the result. A
    command that outlasts timeout seconds raises subprocess.TimeoutExpired."""
    result = subprocess.run([str(a) for a in args], cwd=ROOT, text=True, capture_output=True, timeout=timeout)
    if check and result.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, args))} failed:\n{result.stderr}")
    return result


def submit(label, *args):
    """Submits a job with -wait, records its id under label for section 12,
    and returns its status reply."""
    reply = run(MR, "submit", "-wait", *args).stdout
    job, state = reply.splitlines()[0].split("\t")
    assert state == "SUCCEEDED", reply
    JOBS[job] = label
    return reply


def cat(path):
    """A job's output, its part files in order, as bytes."""
    return subprocess.run([MR, "cat", path], cwd=ROOT, capture_output=True, check=True).stdout


def digest(path):
    """A SHA-256 of a job's part files, in order, read from the host's data/
    directory in chunks (a 1 GiB output does not fit a notebook's memory)."""
    h = hashlib.sha256()
    for part in sorted((ROOT / path.removeprefix("/")).glob("part-r-*")):
        with part.open("rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
    return h.hexdigest()


SORT_BYTES = -(-(2**30) // 100) * 100  # bench/gen-sort.py writes whole 100-byte records


def sort_input():
    """Generates the 1 GiB sort input on first use. It is written to a
    temporary file and renamed when complete, and a file of the wrong size,
    such as one left by an interrupted run, is generated again."""
    if SORT_INPUT.exists() and SORT_INPUT.stat().st_size == SORT_BYTES:
        return
    free = shutil.disk_usage(ROOT / "data").free
    assert free > 3 * 2**30, f"{free / 2**30:.1f} GiB free: the sort needs 3 GiB"
    partial = SORT_INPUT.with_name(SORT_INPUT.name + ".partial")
    with partial.open("wb") as out:
        subprocess.run(["python3", "bench/gen-sort.py", str(2**30), "42"], cwd=ROOT, stdout=out, check=True)
    assert partial.stat().st_size == SORT_BYTES, f"{partial} is {partial.stat().st_size} bytes, not {SORT_BYTES}"
    partial.replace(SORT_INPUT)


def live_workers():
    """The number of workers the coordinator counts as live, or 0."""
    try:
        return next((int(line.split()[1]) for line in metrics_lines() if line.startswith("mr_workers ")), 0)
    except (urllib.error.URLError, OSError):
        return 0


def wait_for_workers(n=4, seconds=60):
    """Waits until n workers are live, and returns whether they are."""
    for _ in range(seconds):
        if live_workers() >= n:
            return True
        time.sleep(1)
    return False


def style(axis, title):
    """The project's chart style: a left-aligned finding as the title,
    hairline axes, no box."""
    axis.set_title(title, loc="left", fontsize=11, color=INK)
    for side in ("top", "right"):
        axis.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        axis.spines[side].set_color("#c3c2b7")
    axis.tick_params(colors="#c3c2b7", labelcolor=INK2)


def status_json():
    """The coordinator's status page as data: its jobs and live workers."""
    with urllib.request.urlopen(STATUS, timeout=5) as response:
        return json.load(response)


def containers():
    """Each running container's name, by its short ID, which is also the
    hostname a worker registers with."""
    lines = run("docker", "ps", "--format", "{{.ID}} {{.Names}}").stdout.splitlines()
    return dict(line.split(" ", 1) for line in lines)


def table(header, rows):
    """Shows rows as a Markdown table, which Jupyter renders."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def metrics_lines():
    """The coordinator's /metrics, as text lines."""
    with urllib.request.urlopen(METRICS, timeout=5) as response:
        return response.read().decode().splitlines()


def fields(text):
    """A status reply, as a dict: job-1\tSUCCEEDED, map\t4/4, counters."""
    return dict(line.split("\t", 1) for line in text.splitlines() if "\t" in line)


assert MR.exists(), "bin/mr is missing: run `make cluster-up mr` first"
try:
    metrics_lines()
except (urllib.error.URLError, OSError) as error:
    raise RuntimeError(f"the coordinator is not answering at {METRICS}: run `make cluster-up`") from error
# A job waits forever for a worker, so stop here if none is live.
workers = live_workers()
if workers == 0:
    raise RuntimeError("the coordinator has no live workers: run `make cluster-up`")
print(f"repository: {ROOT}, live workers: {workers}")


## 2. Check the cluster

One coordinator and four workers, each limited to one CPU. Docker lists the
containers; the coordinator's own view, from its status page, lists the
workers it has registered under its own IDs, which container each is, and
how long ago each last sent a heartbeat. A
worker silent for 10 seconds is declared lost (section 10).


In [ ]:
ps = run(*COMPOSE, "ps", "--format", "{{.Service}}\t{{.Status}}")
print(ps.stdout or "no containers: run `make cluster-up mr`")
print(next((line for line in metrics_lines() if line.startswith("mr_workers ")), "no mr_workers metric"))

status, names = status_json(), containers()
table(
    ["Worker", "Container", "Task slots", "Last heartbeat", "Running"],
    [
        (w["id"], names.get(w["host"], w["host"]), w["slots"], f'{status["at"] - w["last_seen"]:.1f} s ago', len(w["running"]))
        for w in sorted(status["workers"], key=lambda w: w["id"])
    ],
)

## 3. Upload the corpus

Every container sees the host's `data/` directory at `/data`.


In [ ]:
run(MR, "upload", "testdata/corpus/*.txt", CORPUS)
print(sorted(p.name for p in (ROOT / CORPUS.removeprefix("/")).iterdir()))


## 4. Submit word count

The compiled `wordcount` program, by name: 4 map tasks and 4 reduce tasks.
`-wait` polls the job's status until it ends, and the reply ends with the
job's counters, the paper's §4.9: records and bytes read by the map tasks,
emitted by them, folded by the combiner, and written by the reduce tasks.


In [ ]:
WC = f"/data/out/notebook-wc-{RUN}"
reply = submit("word count, compiled", "-m", "4", "-r", "4", "wordcount", WC, f"{CORPUS}/*.txt")
print(reply)
status = fields(reply)
print({key: status[key] for key in ("map", "reduce", "seconds")})


## 5. Read the output


In [ ]:
output = cat(WC)
words = [line.split("\t") for line in output.decode().splitlines()]
top = sorted(((word, int(count)) for word, count in words), key=lambda pair: pair[1], reverse=True)[:10]
print(f"{len(words):,} distinct words")
for word, count in top:
    print(f"{word:>16} {count:>8,}")


## 6. Plot the ten most frequent words


In [ ]:
import matplotlib.pyplot as plt

words_top, counts_top = [w for w, _ in reversed(top)], [c for _, c in reversed(top)]
figure, axis = plt.subplots(figsize=(7, 3.6))
axis.barh(words_top, counts_top, color=BLUE, height=0.6)
for y, count in enumerate(counts_top):
    axis.annotate(f"{count:,}", (count, y), xytext=(4, 0), textcoords="offset points", va="center", color=INK2, fontsize=9)
axis.set_xlabel("Occurrences in the test corpus", color=INK2)
axis.set_xlim(0, counts_top[-1] * 1.12)
axis.grid(axis="x", color="#e1e0d9", linewidth=1)
axis.set_axisbelow(True)
style(axis, f'"{top[0][0]}" alone is {top[0][1] / sum(int(c) for _, c in words):.1%} of all words')
figure.tight_layout()
plt.show()


## 7. The same job as a job file

`examples/wordcount.go` is word count as Go source. It is sent with the job
and interpreted by the workers (ADR 0015): nothing is compiled or uploaded.
Its output must be the compiled program's, byte for byte.


In [ ]:
JOBFILE = f"/data/out/notebook-jobfile-{RUN}"
jobfile = submit("word count, job file", "-m", "4", "-r", "4", "examples/wordcount.go", JOBFILE, f"{CORPUS}/*.txt")
print({key: value for key, value in fields(jobfile).items() if key in ("map", "reduce", "seconds")})
assert cat(JOBFILE) == output, "the job file and the compiled program differ"
print("job file and compiled program agree: True")


## 8. The combiner: fewer bytes shuffled, the same output

The combiner (§4.3) runs the job's `Combine` on each map task's sorted
output, so a word that occurs a thousand times in a split crosses the
network once, not a thousand times. Here the job file runs again with
`-combiner=false`: the output must not change, and `map.output.bytes`, the
bytes every reducer must fetch, shows what the combiner saves. The saving
grows with how often a word repeats within one map task's input: this
corpus is under 0.5 MB in four splits, so expect a few times fewer bytes
here. At 1 GiB the saving is 15× to 78× ([E1](../bench/README.md#evaluation-e0e7)).


In [ ]:
NOCOMB = f"/data/out/notebook-nocombiner-{RUN}"
on = fields(jobfile)  # section 7's job, combiner on
off = fields(submit("word count, job file, no combiner", "-m", "4", "-r", "4", "-combiner=false", "examples/wordcount.go", NOCOMB, f"{CORPUS}/*.txt"))
assert cat(NOCOMB) == output, "the combiner changed the output"
saved = int(off["map.output.bytes"]) / int(on["map.output.bytes"])
for key in ("map.output.bytes", "reduce.input.records", "seconds"):
    print(f"{key:>22}  on {on[key]:>10}   off {off[key]:>10}")
print(f"the combiner shuffles {saved:.1f}× fewer bytes; the output is identical: True")

figure, axis = plt.subplots(figsize=(7, 2.2))
values = [int(on["map.output.bytes"]) / 1e3, int(off["map.output.bytes"]) / 1e3]
axis.barh(["combiner off", "combiner on"], values[::-1], color=[ORANGE, BLUE], height=0.55)
for y, v in enumerate(values[::-1]):
    axis.annotate(f"{v:,.0f} kB", (v, y), xytext=(4, 0), textcoords="offset points", va="center", color=INK2, fontsize=9)
axis.set_xlabel("Bytes shuffled to the reducers (kB)", color=INK2)
axis.set_xlim(0, max(values) * 1.15)
style(axis, f"The combiner shuffles {saved:.1f}× fewer bytes for the same answer")
figure.tight_layout()
plt.show()


## 9. A second program: the most requested pages


In [ ]:
assert (ROOT / "data/input/weblog/access.log").exists(), "run `make upload-example` first"
URLS = f"/data/out/notebook-urls-{RUN}"
submit("URL access frequency", "-m", "2", "urlfreq", URLS, "/data/input/weblog/access.log")
urls = [line.split("\t") for line in cat(URLS).decode().splitlines()]
for url, count in sorted(((url, int(n)) for url, n in urls), key=lambda pair: pair[1], reverse=True)[:5]:
    print(f"{count:>8,}  {url}")


## 10. Kill a worker in the middle of a job

The coordinator notices a dead worker after a 10-second heartbeat timeout,
so this uses a job that lasts longer: a 1 GiB sort with 64 map tasks. Five
seconds in, while the job is still running, `docker kill` stops worker 4.
The job still ends `SUCCEEDED`, and the cell requires the coordinator's log
line that this worker, by the ID it registered with, was lost, and how many
of its tasks ran again elsewhere (§3.3 of the paper). The cell then restarts the worker and waits for all
four to be live.


In [ ]:
SORT = f"/data/out/notebook-sort-kill-{RUN}"
if not KILL_WORKER:
    print("skipped: KILL_WORKER is False")
else:
    sort_input()
    names = containers()
    killed = next((w["id"] for w in status_json()["workers"] if names.get(w["host"]) == "mapreduce-worker-4"), None)
    assert killed, "mapreduce-worker-4 is not registered with the coordinator: check `docker compose ps`"
    reply = None
    job = subprocess.Popen(
        [MR, "submit", "-m", "64", "-r", "4", "-wait", "distsort", SORT, "/data/input/sort-1g.txt"],
        cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    try:
        time.sleep(5)
        assert job.poll() is None, "the sort finished within 5 s: no worker was lost mid-job"
        since = run("date", "-u", "+%Y-%m-%dT%H:%M:%SZ").stdout.strip()
        run("docker", "kill", "mapreduce-worker-4")
        print(f"killed mapreduce-worker-4, the coordinator's {killed}, five seconds into the job")
        reply, errors = job.communicate()
    finally:
        if reply is None:  # an error above: do not leave the client running
            job.kill()
        run(*COMPOSE, "up", "-d", "--no-build", "worker")
    assert wait_for_workers(), "fewer than 4 live workers after 60 s: check `docker compose ps`"
    assert job.returncode == 0, errors
    status = fields(reply)
    JOBS[reply.split("\t")[0]] = "1 GiB sort, worker 4 killed"
    print(reply.splitlines()[0], {key: status[key] for key in ("map", "reduce", "seconds")})
    assert reply.splitlines()[0].endswith("SUCCEEDED"), reply
    assert status["reduce.output.records"] == status["map.input.records"], "records lost"
    logs = run("docker", "logs", "--since", since, "mapreduce-coordinator-1")
    lost = [line for line in (logs.stdout + logs.stderr).splitlines() if "worker lost" in line and f"worker={killed} " in line]
    assert lost, f"the coordinator did not log losing {killed}, the killed worker"
    reset = int(lost[-1].split("tasks_reset=")[1].split()[0])
    assert reset > 0, lost[-1]
    print(lost[-1])
    print(
        f"The coordinator lost {killed}, the killed worker, and reset {reset} of its tasks; other workers ran them again,"
        f" and the job still wrote all {int(status['reduce.output.records']):,} records in {float(status['seconds']):.1f} s."
    )


## 11. Restart the coordinator in the middle of a job

The paper's master is a single point of failure: if it dies, the job is
aborted (§3.3). Here the coordinator writes every decision to a write-ahead
log before it replies ([ADR 0007](../docs/design/0007-write-ahead-log.md)),
so a restarted coordinator replays the log and finishes the job.

The same 1 GiB sort is submitted without `-wait`. Six seconds in, while the
job is still running, `docker kill` crashes the coordinator and
`docker start` brings it back, as `make e2e` does. (A `docker restart`
would stop it cleanly, and a clean stop tells idle workers to exit.) The job ends
`SUCCEEDED`, and its output is byte for byte the output of section 10's run.
On the course VM a restart costs about 1.2 s
([E4](../bench/README.md#evaluation-e0e7)).


In [ ]:
RESTARTED = f"/data/out/notebook-sort-restart-{RUN}"
if not RESTART_COORDINATOR:
    print("skipped: RESTART_COORDINATOR is False")
else:
    sort_input()
    started = time.monotonic()
    job_id = run(MR, "submit", "-m", "64", "-r", "4", "distsort", RESTARTED, "/data/input/sort-1g.txt").stdout.split()[0]
    time.sleep(6)
    before = run(MR, "status", job_id, timeout=10).stdout.splitlines()[0]
    print(before, "<- before the restart")
    assert before.split("\t")[1] in ("PENDING", "MAPPING", "REDUCING"), "the job ended before the restart"
    run("docker", "kill", "mapreduce-coordinator-1")
    run("docker", "start", "mapreduce-coordinator-1")
    print("crashed and started mapreduce-coordinator-1")
    state, reply = "", None
    while time.monotonic() - started < 300:
        try:  # Unavailable while it restarts; a hung request counts as no answer
            reply = run(MR, "status", job_id, check=False, timeout=10)
        except subprocess.TimeoutExpired:
            reply = None
        state = reply.stdout.split("\t", 1)[1].split("\n", 1)[0] if reply and reply.returncode == 0 else ""
        if state in ("SUCCEEDED", "FAILED"):
            break
        time.sleep(1)
    JOBS[job_id] = "1 GiB sort, coordinator restarted"
    print(f"{job_id} {state} after {time.monotonic() - started:.0f} s")
    assert state == "SUCCEEDED", (reply.stdout + reply.stderr) if reply else "no status reply"
    assert wait_for_workers(), "fewer than 4 live workers after 60 s: check `docker compose ps`"
    if KILL_WORKER:
        assert digest(RESTARTED) == digest(SORT), "the restarted job's output differs"
        print("same output as the run with a worker killed: True")


## 12. The coordinator's metrics

Every job this run submitted, as the coordinator reports it on its status
page (state, tasks, time and backup tasks, §3.6), then the raw Prometheus
metrics the status page and the Grafana dashboard are drawn from: live
workers, and completed tasks per job.


In [ ]:
jobs = {job["id"]: job for job in status_json()["jobs"]}
table(
    ["Job", "What it ran", "State", "Map tasks", "Reduce tasks", "Seconds", "Backups"],
    [
        (
            job,
            label,
            jobs[job]["state"],
            f'{jobs[job]["map_completed"]}/{jobs[job]["map_total"]}',
            f'{jobs[job]["reduce_completed"]}/{jobs[job]["reduce_total"]}',
            f'{jobs[job]["duration_seconds"]:.1f}',
            jobs[job]["backup_attempts"],
        )
        for job, label in JOBS.items()
        if job in jobs
    ],
)
for line in metrics_lines():
    if line.startswith("mr_workers ") or ('state="completed"' in line and any(f'job="{job}"' in line for job in JOBS)):
        print(line)

## 13. Local mode is the oracle

Local mode runs the same job in one process, without Docker. Its output, the
cluster's and the expected output checked in at `testdata/corpus/expected`
must all be the same, byte for byte (README, "Run a job on one machine").


In [ ]:
LOCAL = f"data/out/notebook-local-{RUN}"
run(MR, "run", "-local", "-m", "4", "-r", "4", "wordcount", LOCAL, "testdata/corpus/*.txt")
assert cat(f"/{LOCAL}") == output, "local mode and the cluster differ"
print("local mode and the cluster agree: True")
assert output == EXPECTED, "the cluster's output differs from testdata/corpus/expected"
print("the cluster matches testdata/corpus/expected: True")


## 14. The evaluation at a glance

The paper's §5 experiments and our own, rerun on the course VM with 1 GiB
inputs: medians of five runs, every run's output checked against an
undisturbed run's. `bench/plot-experiments.py` draws them from the data in
`bench/evaluation/`, in order E0 to E7, with the fault recap after E4; [bench/README.md](../bench/README.md#evaluation-e0e7)
has every figure, table and command.


In [ ]:
THEME = ""  # "-dark" for the dark variants
# Each figure's finding, as README.md's results table states it.
FIGURES = {
    "E0": "E0, scaling over workers: word count runs 3.7× faster on 4 workers than on 1, distributed sort 3.1×, grep 2.0×.",
    "E1": "E1, the combiner: 15× to 78× fewer bytes shuffled and 16–42% less job time, more as the keys get more skewed.",
    "E2": "E2, backup tasks: with one worker at 0.2 CPU, backups cut 17.9 s to 11.4 s; with no straggler they cost nothing measurable.",
    "E3": "E3, losing a worker: every job finishes with the undisturbed run's output, +8 s to +26 s in all.",
    "E4": "E4, a coordinator restart: recovery from the log costs 1.2 s (15%) and keeps every finished task.",
    "faults": "E2 to E4 together: what each fault costs.",
    "E5": "E5, an interpreted job file: 2.3× slower than compiled with the helpers api.Words/api.Sum, 10.2× without them.",
    "E6": "E6, the combiner on real text: 5.2× fewer bytes and 11% less job time on crawled English.",
    "E7": "E7, the paper's backups against LATE: either rule cuts a sort on mixed-speed workers by 32%; LATE needs fewer backups, and the two times overlap.",
}
for name, finding in FIGURES.items():
    display(Markdown(f"**{finding}**"))
    display(Image(filename=str(ROOT / "bench/evaluation" / f"{name}{THEME}.png"), width=760))

## 15. If a cell fails

Every check stops with a message that names its cause. Rerunning from the
top is always safe: setup clears what an interrupted run left.

| Message or symptom | What to run |
|---|---|
| `bin/mr is missing`, `not answering`, `no live workers` | `make cluster-up mr`, then `docker compose -p mapreduce -f deploy/compose/compose.yaml ps` |
| `fewer than 4 live workers after 60 s` | `docker compose -p mapreduce -f deploy/compose/compose.yaml up -d --no-build worker` |
| ``run `make upload-example` first`` | `make upload-example` |
| `GiB free: the sort needs 3 GiB` | free disk space, or set `KILL_WORKER` and `RESTART_COORDINATOR` to `False` |
| `the sort finished within 5 s`, `the job ended before the restart` | the machine is faster than the course VM: rerun the cell, or skip it with its flag |
| Anything else | `make demo-check`: one `OK`, `WARN` or `FAIL` line per thing the demo needs, each `FAIL` with what to run |


## 16. Clean up

This run's input and output. The sort input stays in `data/input/` for the next run.


In [ ]:
for path in (WC, JOBFILE, NOCOMB, URLS, SORT, RESTARTED, f"/{LOCAL}", CORPUS):
    shutil.rmtree(ROOT / path.removeprefix("/"), ignore_errors=True)
print("removed this run's input and output")
